In [31]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

In [32]:
print(df_train.columns)

Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [35]:
df_train_merged["Race"].unique()

array(['Mexico City Grand Prix', 'Italian Grand Prix',
       'Monaco Grand Prix', 'Azerbaijan Grand Prix',
       'São Paulo Grand Prix', 'Emilia Romagna Grand Prix',
       'Canadian Grand Prix', 'Chinese Grand Prix',
       'Singapore Grand Prix', 'Hungarian Grand Prix',
       'French Grand Prix', 'Abu Dhabi Grand Prix', 'Austrian Grand Prix',
       'Japanese Grand Prix', 'Belgian Grand Prix', 'British Grand Prix',
       'Dutch Grand Prix', 'United States Grand Prix', 'Miami Grand Prix',
       'Spanish Grand Prix', 'Saudi Arabian Grand Prix',
       'Las Vegas Grand Prix', 'Qatar Grand Prix',
       'Australian Grand Prix', 'Bahrain Grand Prix',
       'Pre-Season Testing', 'Pre-Season Track Session',
       'Pre-Season Test'], dtype=object)

In [36]:
df_train_merged.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')

In [37]:
df_train_merged[["Compound"]].value_counts()

Compound    
MEDIUM          248780
HARD            215485
SOFT             51488
INTERMEDIATE     22938
WET               1754
Name: count, dtype: int64

In [38]:
#Tire dataset
def add_tire_life_columns(df):
    max_tire_life = {
        "MEDIUM": 40,
        "HARD": 55,
        "SOFT": 30,
        "INTERMEDIATE": 25,
        "WET": 20
    }

    df["MaxTireLife"] = df["Compound"].map(max_tire_life)

    df["TireRemainingLife"] = (
        df["MaxTireLife"] - df["TyreLife"]
    ).clip(lower=0)
    df["TireAgePct"] = (
    df["TyreLife"] / df["MaxTireLife"]).clip(0, 1)
    df["TireRemainingPct"] = (df["TireRemainingLife"] / df["MaxTireLife"]).clip(0, 1)
    

    return df



In [39]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [40]:
def add_laps_for_each_race(df):
    df["RaceLaps"] = (
    df["LapNumber"] / (df["RaceProgress"])
).round()
    df["LapsRemaining"] = df["RaceLaps"] - df["LapNumber"]
    return df


In [41]:
def prev_position_calcute(df):
    df["Prev_Position"] = df["Position"] + df["Position_Change"]
    return df


In [42]:
def get_trend_data(df):
    df["LapTimeAvg_3"] = (
        df.groupby(["Year", "Race", "Driver"])["LapTime (s)"]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )
    return df


In [43]:
def set_fe(df):
    df = add_tire_life_columns(df)
    df = add_laps_for_each_race(df)
    df= prev_position_calcute(df)
    df = get_trend_data(df)
    return df

In [14]:
def calculate_race_progress(df, is_drop=True):
    condition = (df["Race"] == "Pre-Season Testing") & (
        (df["RaceLaps"] < 31) | (df["RaceLaps"] > 78)
    )

    if is_drop:
        # Keep rows that do NOT match the condition
        df = df[~condition] 
    F1_RACE_LAPS = {
        "Bahrain Grand Prix": 57,
        "Saudi Arabian Grand Prix": 50,
        "Australian Grand Prix": 58,
        "Azerbaijan Grand Prix": 51,
        "Miami Grand Prix": 57,
        "Monaco Grand Prix": 78,
        "Spanish Grand Prix": 66,
        "Canadian Grand Prix": 70,
        "Austrian Grand Prix": 71,
        "British Grand Prix": 52,
        "Hungarian Grand Prix": 70,
        "Belgian Grand Prix": 44,
        "Dutch Grand Prix": 72,
        "Italian Grand Prix": 51,
        "Singapore Grand Prix": 62,
        "Japanese Grand Prix": 53,
        "Qatar Grand Prix": 57,
        "United States Grand Prix": 56,
        "Mexico City Grand Prix": 71,
        "São Paulo Grand Prix": 71,
        "Las Vegas Grand Prix": 50,
        "Abu Dhabi Grand Prix": 58,
        }
    mask = df["Race"].isin(F1_RACE_LAPS.keys())

    # Update only matching rows
    df.loc[mask, "RaceLaps"] = df.loc[mask, "Race"].map(F1_RACE_LAPS)
    df["RaceLaps"] = df["Race"].map(F1_RACE_LAPS)
    df["RaceProgress"] = df["LapNumber"] / df["RaceLaps"]
    df["LapsRemaining"] = df["RaceLaps"] - df["LapNumber"]
    return df

In [44]:
# df_external_fe =set_fe(df_external.copy())
df_train_fe =set_fe(df_train.copy())
df_train_merged_fe =set_fe(df_train_merged.copy())
df_test_fe =set_fe(df_test.copy())

In [ ]:
# df_train_fe =calculate_race_progress(df_train_fe)
# df_train_merged_fe =calculate_race_progress(df_train_merged_fe)
# df_test_fe =calculate_race_progress(df_test_fe,is_drop=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24560\1365074506.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["RaceLaps"] = df["Race"].map(F1_RACE_LAPS)
C:\Users\Admin\AppData\Local\Temp\ipykernel_24560\1365074506.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["RaceProgress"] = df["LapNumber"] / df["RaceLaps"]
C:\Users\Admin\AppData\Local\Temp\ipykernel_24560\1365074506.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_index

In [45]:
df_train =df_train.drop("id",axis=1)
df_test =df_test.drop("id",axis=1)

In [46]:
df_train_fe =df_train_fe.drop("id",axis=1)
df_test_fe =df_test_fe.drop("id",axis=1)

In [47]:
df_train_merged.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [48]:
df_train_merged.nunique().sort_values(ascending=False)

Cumulative_Degradation    163914
LapTime_Delta              73068
LapTime (s)                45108
RaceProgress                1947
Driver                       887
TyreLife                      79
LapNumber                     78
Position_Change               37
Race                          28
Position                      20
Stint                          8
Compound                       5
Year                           4
PitStop                        2
PitNextLap                     2
dtype: int64

In [49]:
df_train_merged['Normalized_TyreLife'] = df_train_merged['TyreLife'] / df_train_merged.groupby(
    ['Year', 'Race', 'Driver', 'Stint']
)['TyreLife'].transform('max')

In [50]:
df_test['Normalized_TyreLife'] = df_test['TyreLife'] / df_test.groupby(['Year', 'Race', 'Driver', 'Stint'])['TyreLife'].transform('max')

In [80]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),...,PitNextLap,MaxTireLife,TireRemainingLife,TireAgePct,TireRemainingPct,RaceLaps,LapsRemaining,Prev_Position,LapTimeAvg_3,Normalized_TyreLife
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,...,0.0,40,34.0,0.150000,0.850000,71.0,65.0,12.0,83.921,0.222222
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,...,1.0,55,38.0,0.309091,0.690909,77.0,53.0,6.0,83.845,0.566667
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,...,0.0,40,17.0,0.575000,0.425000,76.0,53.0,9.0,79.239,0.389831
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,...,1.0,55,22.0,0.600000,0.400000,72.0,22.0,14.0,87.076,1.000000
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,...,0.0,55,6.0,0.890909,0.109091,75.0,26.0,12.0,78.328,0.710145


In [ ]:
# df_train_merged["Cumulative_Degradation"] = df_train_merged["Cumulative_Degradation"].fillna(0).astype("int")
# cols = ["LapTimeAvg_3",
# "Cumulative_Degradation",
# "LapTime_Delta",
# "LapTime (s)"]
# df_train_merged[cols] = df_train_merged[cols].round(1)
# df_train_merged["RaceProgress"] = df_train_merged["RaceProgress"].round(2)



In [ ]:
# df_test["Cumulative_Degradation"] = df_test["Cumulative_Degradation"].fillna(0).astype("int")
# cols = ["LapTimeAvg_3",
# "Cumulative_Degradation",
# "LapTime_Delta",
# "LapTime (s)"]
# df_test[cols] = df_test[cols].round(1)
# df_test["RaceProgress"] = df_train_merged["RaceProgress"].round(2)

In [ ]:
# df_train_merged[["LapTimeAvg_3",
# "Cumulative_Degradation",
# "LapTime_Delta",
# "LapTime (s)",
# "RaceProgress"]].head()

,LapTimeAvg_3,Cumulative_Degradation,LapTime_Delta,LapTime (s),RaceProgress
0,83.9,-10,-21.2,83.9,0.08
1,83.8,-33,-22.9,83.8,0.31
2,79.2,-12,0.1,79.2,0.30
3,87.1,-31,-13.9,87.1,0.69
4,78.3,-33,-0.5,78.3,0.65


In [ ]:
# df_train_merged.nunique().sort_values(ascending=False)

LapTime_Delta             1606
LapTime (s)                942
LapTimeAvg_3               914
Driver                     887
Cumulative_Degradation     463
TireAgePct                 130
TireRemainingPct           130
RaceLaps                   123
LapsRemaining              117
RaceProgress               100
TyreLife                    79
LapNumber                   78
TireRemainingLife           55
Prev_Position               50
Position_Change             37
Race                        28
Position                    20
Stint                        8
Compound                     5
MaxTireLife                  5
Year                         4
PitStop                      2
PitNextLap                   2
dtype: int64

In [47]:
df_test.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
MaxTireLife               0
TireRemainingLife         0
TireAgePct                0
TireRemainingPct          0
RaceLaps                  0
LapsRemaining             0
Prev_Position             0
LapTimeAvg_3              0
dtype: int64

In [51]:
from autogluon.tabular import TabularDataset, TabularPredictor
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [52]:
print(df_train.columns)
print(df_train.columns)

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')
Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')


In [53]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [54]:
models = {
    # --- Existing Gradient Boosted Trees ---
    "GBM": [
        {},  # Standard LightGBM
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # ExtraTrees LightGBM
    ],
    "XGB": {},  # XGBoost
    "CAT": {},  # CatBoost

    # --- TOP ADDITION 1: PyTorch Neural Network ---
    "NN_TORCH": {},  # Generates NeuralNetTorch_BAG_L1

    # --- TOP ADDITION 2: Extra Trees (Gini & Entropy) ---
    "XT": [
        {"criterion": "gini", "ag_args": {"name_suffix": "Gini"}},
        {"criterion": "entropy", "ag_args": {"name_suffix": "Entr"}},
    ],
}


In [ ]:
predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc', path='ag_models15').fit(
    train_data=df_train_merged_fe,
    ag_args_fit={"num_gpus": 1},
    time_limit=3600*9,
    presets='best_quality',
    verbosity=3,
    num_stack_levels=1,
    hyperparameters=models,
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.8.0+cu129
CUDA Version:       12.9
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.60 GB / 15.06 GB (30.5%)
Disk Space Avail:   647.19 GB / 930.47 GB (69.6%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 1,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 '

[50]	valid_set's binary_logloss: 0.283978
[100]	valid_set's binary_logloss: 0.260726
[150]	valid_set's binary_logloss: 0.251852
[200]	valid_set's binary_logloss: 0.246249
[250]	valid_set's binary_logloss: 0.243079
[300]	valid_set's binary_logloss: 0.240699
[350]	valid_set's binary_logloss: 0.238587
[400]	valid_set's binary_logloss: 0.237128
[450]	valid_set's binary_logloss: 0.235646
[500]	valid_set's binary_logloss: 0.234578
[550]	valid_set's binary_logloss: 0.233693
[600]	valid_set's binary_logloss: 0.232974
[650]	valid_set's binary_logloss: 0.232242
[700]	valid_set's binary_logloss: 0.231662
[750]	valid_set's binary_logloss: 0.231113
[800]	valid_set's binary_logloss: 0.230736
[850]	valid_set's binary_logloss: 0.230197
[900]	valid_set's binary_logloss: 0.229975
[950]	valid_set's binary_logloss: 0.229565
[1000]	valid_set's binary_logloss: 0.229264
[1050]	valid_set's binary_logloss: 0.228929
[1100]	valid_set's binary_logloss: 0.228678
[1150]	valid_set's binary_logloss: 0.22847
[1200]	va

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.283521
[100]	valid_set's binary_logloss: 0.261787
[150]	valid_set's binary_logloss: 0.252812
[200]	valid_set's binary_logloss: 0.247872
[250]	valid_set's binary_logloss: 0.243723
[300]	valid_set's binary_logloss: 0.240709
[350]	valid_set's binary_logloss: 0.238279
[400]	valid_set's binary_logloss: 0.236717
[450]	valid_set's binary_logloss: 0.235286
[500]	valid_set's binary_logloss: 0.234188
[550]	valid_set's binary_logloss: 0.233261
[600]	valid_set's binary_logloss: 0.232232
[650]	valid_set's binary_logloss: 0.231411
[700]	valid_set's binary_logloss: 0.230678
[750]	valid_set's binary_logloss: 0.230063
[800]	valid_set's binary_logloss: 0.229524
[850]	valid_set's binary_logloss: 0.229003
[900]	valid_set's binary_logloss: 0.228463
[950]	valid_set's binary_logloss: 0.227981
[1000]	valid_set's binary_logloss: 0.227657
[1050]	valid_set's binary_logloss: 0.227204
[1100]	valid_set's binary_logloss: 0.226903
[1150]	valid_set's binary_logloss: 0.226738
[1200]	v

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


...

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.45368
[50]	validation_0-logloss:0.19921
[100]	validation_0-logloss:0.19893
[137]	validation_0-logloss:0.19904


	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.45343
[50]	validation_0-logloss:0.19582
[100]	validation_0-logloss:0.19537
[150]	validation_0-logloss:0.19547
[172]	validation_0-logloss:0.19554


Saving c:\Darshak\Projects\Hackathon\ag_models14\models\XGBoost_BAG_L2\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models14\models\XGBoost_BAG_L2\model.pkl
	0.9627	 = Validation score   (roc_auc)
	35.61s	 = Training   runtime
	0.76s	 = Validation runtime
	2467.9	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models14\models\trainer.pkl
Fitting model: NeuralNetTorch_BAG_L2 ... Training model for up to 20254.59s of the 20254.56s of remaining time.
	Fitting NeuralNetTorch_BAG_L2 with 'num_gpus': 1, 'num_cpus': 16
Saving c:\Darshak\Projects\Hackathon\ag_models14\models\NeuralNetTorch_BAG_L2\utils\model_template.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\NeuralNetTorch_BAG_L2\utils\model_template.pkl
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=1)
	Fit constraints were validated upstream on the full training data, skipping...
Tabular Neural Network 

In [56]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,0.963153,roc_auc,39.040957,6911.982308,0.049794,14.904287,3,True,14
1,CatBoost_BAG_L2,0.963068,roc_auc,26.842516,6107.423786,0.229042,101.455078,2,True,9
2,XGBoost_BAG_L2,0.962663,roc_auc,27.374064,6041.580454,0.760590,35.611746,2,True,12
3,LightGBMLarge_BAG_L2,0.962447,roc_auc,27.151249,6024.026470,0.537774,18.057762,2,True,8
4,ExtraTreesEntr_BAG_L2,0.962291,roc_auc,37.041260,6028.494108,10.427785,22.525400,2,True,11
5,NeuralNetTorch_BAG_L2,0.962266,roc_auc,28.066413,6759.773321,1.452938,753.804613,2,True,13
6,LightGBM_BAG_L2,0.962231,roc_auc,26.881398,6019.292931,0.267923,13.324223,2,True,7
7,ExtraTreesGini_BAG_L2,0.961966,roc_auc,36.283506,6023.864432,9.670032,17.895724,2,True,10
8,WeightedEnsemble_L2,0.961841,roc_auc,8.246526,5703.422764,0.050192,6.240369,2,True,6
9,CatBoost_BAG_L1,0.958953,roc_auc,3.044774,812.774353,3.044774,812.774353,1,True,3


In [ ]:
# predictor = TabularPredictor.load(f"ag_models13")
# leaderboard = predictor.leaderboard()
# leaderboard

In [22]:
df_test.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [57]:
df=predictor.predict_proba(df_test_fe)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\CatBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\NeuralNetTorch_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\CatBoost_BAG_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\ExtraTreesEntr_BAG_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\LightGBM_BAG_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\NeuralNetTorch_BAG_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models14\models\WeightedEnsemble_L3\model.pkl


,0,1
0,0.996255,0.003745
1,0.994461,0.005539
2,0.995770,0.004230
3,0.823691,0.176309
4,0.095334,0.904666


In [61]:
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [62]:
df_sample_out['PitNextLap']

0         0
1         0
2         0
3         0
4         0
         ..
188160    0
188161    0
188162    0
188163    0
188164    0
Name: PitNextLap, Length: 188165, dtype: int64

In [63]:
df_sample_out.count()

id            188165
PitNextLap    188165
dtype: int64

In [65]:
df_sample_out['PitNextLap']=df[1]

In [66]:
# df_sample_out.loc[df_sample_out['PitNextLap'].isna(), 'PitNextLap'] = 0.0

In [67]:
df_sample_out.head()

,id,PitNextLap
0,439140,0.003745
1,439141,0.005539
2,439142,0.004230
3,439143,0.176309
4,439144,0.904666


In [68]:
df_sample_out.to_csv("My_output/all_model_together_best_with_2_extra_models_normalizetl_fe_7.csv", index=False)

In [ ]:
#Try without driver

In [95]:
df_train_merged_wo_driver = df_train_merged.drop("Driver", axis=1)
df_train_wo_driver = df_train.drop("Driver", axis=1)
df_test_wo_driver = df_test.drop("Driver", axis=1)

In [96]:
predictor_wo_driver = TabularPredictor(label=TARGET,eval_metric='roc_auc', path='ag_models13').fit(
    train_data=df_train_merged_wo_driver,
    ag_args_fit={"num_gpus": 1},
    time_limit=3600*9,
    presets='best_quality',
    verbosity=3,
    num_stack_levels=0,
    hyperparameters=models,
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.8.0+cu129
CUDA Version:       12.9
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.98 GB / 15.06 GB (33.1%)
Disk Space Avail:   647.13 GB / 930.47 GB (69.5%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 '

[50]	valid_set's binary_logloss: 0.283358
[100]	valid_set's binary_logloss: 0.259077
[150]	valid_set's binary_logloss: 0.248858
[200]	valid_set's binary_logloss: 0.243297
[250]	valid_set's binary_logloss: 0.239875
[300]	valid_set's binary_logloss: 0.236859
[350]	valid_set's binary_logloss: 0.234092
[400]	valid_set's binary_logloss: 0.231986
[450]	valid_set's binary_logloss: 0.230119
[500]	valid_set's binary_logloss: 0.228529
[550]	valid_set's binary_logloss: 0.227317
[600]	valid_set's binary_logloss: 0.226227
[650]	valid_set's binary_logloss: 0.225313
[700]	valid_set's binary_logloss: 0.224299
[750]	valid_set's binary_logloss: 0.223344
[800]	valid_set's binary_logloss: 0.222493
[850]	valid_set's binary_logloss: 0.221834
[900]	valid_set's binary_logloss: 0.221128
[950]	valid_set's binary_logloss: 0.220607
[1000]	valid_set's binary_logloss: 0.22007
[1050]	valid_set's binary_logloss: 0.219543
[1100]	valid_set's binary_logloss: 0.219023
[1150]	valid_set's binary_logloss: 0.218692
[1200]	va

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.284252
[100]	valid_set's binary_logloss: 0.260215
[150]	valid_set's binary_logloss: 0.25003
[200]	valid_set's binary_logloss: 0.244546
[250]	valid_set's binary_logloss: 0.240755
[300]	valid_set's binary_logloss: 0.237883
[350]	valid_set's binary_logloss: 0.235311
[400]	valid_set's binary_logloss: 0.233384
[450]	valid_set's binary_logloss: 0.23183
[500]	valid_set's binary_logloss: 0.230181
[550]	valid_set's binary_logloss: 0.228827
[600]	valid_set's binary_logloss: 0.227761
[650]	valid_set's binary_logloss: 0.226591
[700]	valid_set's binary_logloss: 0.225568
[750]	valid_set's binary_logloss: 0.224623
[800]	valid_set's binary_logloss: 0.223895
[850]	valid_set's binary_logloss: 0.223081
[900]	valid_set's binary_logloss: 0.222201
[950]	valid_set's binary_logloss: 0.221693
[1000]	valid_set's binary_logloss: 0.221129
[1050]	valid_set's binary_logloss: 0.220547
[1100]	valid_set's binary_logloss: 0.220044
[1150]	valid_set's binary_logloss: 0.21958
[1200]	vali

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.286501
[100]	valid_set's binary_logloss: 0.261955
[150]	valid_set's binary_logloss: 0.252519
[200]	valid_set's binary_logloss: 0.247066
[250]	valid_set's binary_logloss: 0.243133
[300]	valid_set's binary_logloss: 0.240279
[350]	valid_set's binary_logloss: 0.238456
[400]	valid_set's binary_logloss: 0.236295
[450]	valid_set's binary_logloss: 0.234466
[500]	valid_set's binary_logloss: 0.232939
[550]	valid_set's binary_logloss: 0.231343
[600]	valid_set's binary_logloss: 0.230143
[650]	valid_set's binary_logloss: 0.229197
[700]	valid_set's binary_logloss: 0.228222
[750]	valid_set's binary_logloss: 0.227372
[800]	valid_set's binary_logloss: 0.226579
[850]	valid_set's binary_logloss: 0.225595
[900]	valid_set's binary_logloss: 0.224974
[950]	valid_set's binary_logloss: 0.224463
[1000]	valid_set's binary_logloss: 0.22383
[1050]	valid_set's binary_logloss: 0.223402
[1100]	valid_set's binary_logloss: 0.222926
[1150]	valid_set's binary_logloss: 0.222412
[1200]	va

KeyboardInterrupt: 

In [169]:
leaderboard = predictor_wo_driver.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.959125,roc_auc,37.932610,1384.518738,0.050312,5.060219,2,True,5
1,LightGBM_BAG_L1,0.958277,roc_auc,11.543802,210.660673,11.543802,210.660673,1,True,1
2,XGBoost_BAG_L1,0.957086,roc_auc,2.971786,273.060256,2.971786,273.060256,1,True,4
3,LightGBMLarge_BAG_L1,0.956886,roc_auc,22.951177,296.346481,22.951177,296.346481,1,True,2
4,CatBoost_BAG_L1,0.954769,roc_auc,0.415533,599.391110,0.415533,599.391110,1,True,3


In [170]:
df=predictor_wo_driver.predict_proba(df_test)
df.head()
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()
df_sample_out['PitNextLap']=df[1]
df_sample_out.to_csv("My_output/all_model_together_best_fe_without_driver_1.csv", index=False)

Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\CatBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\WeightedEnsemble_L2\model.pkl
